# ===============================================================
# Phase 3: Embed & Index into Chroma
#  - Input : data/processed/chunks.jsonl
#  - Output: vectorstore (persist), collection 'jenosize-ideas'
# ===============================================================

In [1]:


from pathlib import Path
from dotenv import load_dotenv
import os, json

load_dotenv()

# 💡 ตั้ง root โปรเจกต์แบบชัด ๆ (แก้ได้นะถ้า path คุณต่างไป)
PROJECT_ROOT = Path(r"D:\mini-jane-demo")

RAW_PATH        = PROJECT_ROOT / "data" / "raw"
PROCESSED_PATH  = PROJECT_ROOT / "data" / "processed"
CHUNKS_FILE     = PROCESSED_PATH / "chunks.jsonl"

# จาก .env หรือ fallback เป็น ./vectorstore
CHROMA_DIR   = Path(os.getenv("CHROMA_PERSIST_DIR", PROJECT_ROOT / "vectorstore"))
EMBED_MODEL  = os.getenv("EMBED_MODEL", "BAAI/bge-m3")   # multilingual แนะนำ

print("PROJECT_ROOT :", PROJECT_ROOT)
print("CHUNKS_FILE  :", CHUNKS_FILE, "exists?", CHUNKS_FILE.exists())
print("CHROMA_DIR   :", CHROMA_DIR)
print("EMBED_MODEL  :", EMBED_MODEL)


PROJECT_ROOT : D:\mini-jane-demo
CHUNKS_FILE  : D:\mini-jane-demo\data\processed\chunks.jsonl exists? True
CHROMA_DIR   : vectorstore
EMBED_MODEL  : BAAI/bge-m3


In [2]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from tqdm import tqdm


In [3]:
# โหลดโมเดลฝังเวกเตอร์ (รองรับไทย/อังกฤษ)
embedder = SentenceTransformer(EMBED_MODEL)

# เตรียม Chroma persistent client
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(CHROMA_DIR), settings=Settings(allow_reset=True))

# ชื่อคอลเลกชัน
COLLECTION_NAME = "jenosize-ideas"
collection = client.get_or_create_collection(name=COLLECTION_NAME)

print("✅ Ready -> collection:", COLLECTION_NAME)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\mini-jane-demo\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ruthello\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Ready -> collection: jenosize-ideas


In [12]:
# เลือกได้: reset ทั้ง collection (ใช้ด้วยความระวัง)
RESET_COLLECTION = True  # เปลี่ยนเป็น True ถ้าอยากล้างก่อน

if RESET_COLLECTION:
    client.delete_collection(COLLECTION_NAME)
    collection = client.get_or_create_collection(name=COLLECTION_NAME)
    print("🧹 Collection reset.")

# โหลด chunks เข้าหน่วยความจำ
assert CHUNKS_FILE.exists(), f"Not found: {CHUNKS_FILE}"
with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

len(chunks), chunks[0] if chunks else None


🧹 Collection reset.


(446,
 {'doc_id': 'https://www.jenosize.com/en/ideas/experience-the-new-world/event-design-thinking',
  'url': 'https://www.jenosize.com/en/ideas/experience-the-new-world/event-design-thinking',
  'title': '5 Steps of Event Design Thinking for Memorable Events',
  'category': 'Experience the New World',
  'section': 'Body',
  'chunk_index': 1,
  'text': "5 Key Steps in the Event Design Thinking Process\nEvents are powerful platforms where businesses can unleash creativity, connect with audiences, and leave a lasting impression. But the real question is: how do you design an event that’s truly unforgettable? One proven approach adopted by leading global agencies and organizations is event design thinking—a human-centered design process that puts the event attendee at the core, focusing on emotional engagement, sensory experience, and business goals. What Is Event Design Thinking ? Event design thinking is a strategic planning method that begins with understanding the needs and motivatio

In [8]:
def batched(lst, size=64):
    buf = []
    for x in lst:
        buf.append(x)
        if len(buf) >= size:
            yield buf
            buf = []
    if buf:
        yield buf

def make_id(item):
    # ป้องกันซ้ำ: ใช้ url + chunk_index เป็น id
    return f"{item['doc_id']}#chunk{item['chunk_index']}"


In [13]:
# A) Diagnose duplicate IDs in memory
from collections import Counter

def make_id(item):
    return f"{item['doc_id']}#chunk{item['chunk_index']}"

ids_all = [make_id(c) for c in chunks]
cnt = Counter(ids_all)
dups = [k for k,v in cnt.items() if v > 1]

print("Total chunks  :", len(ids_all))
print("Unique IDs    :", len(cnt))
print("Duplicate IDs :", len(dups))
if dups:
    print("Example dup IDs:", dups[:5])


Total chunks  : 446
Unique IDs    : 446
Duplicate IDs : 0


In [11]:
# B) Dedupe globally by ID (keep the last occurrence)
uniq = {}
for c in chunks:
    uniq[make_id(c)] = c   # ตัวท้ายทับตัวก่อนหน้า

chunks_dedup = list(uniq.values())
print("After dedupe:", len(chunks_dedup), "unique chunks")

# (ตัวเลือก) เขียนทับไฟล์เดิมหรือเซฟชื่อใหม่
# เขียนทับไฟล์เดิม:
with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    for c in chunks_dedup:
        json.dump(c, f, ensure_ascii=False); f.write("\n")

# อัปเดตค่าที่อยู่ในหน่วยความจำให้เป็นเวอร์ชัน dedup แล้ว
chunks = chunks_dedup


After dedupe: 446 unique chunks


In [14]:
added = 0

for batch in tqdm(list(batched(chunks, size=64)), desc="Indexing to Chroma"):
    ids = [make_id(b) for b in batch]
    docs = [b["text"] for b in batch]
    metas = [{
        "url": b["url"],
        "title": b["title"],
        "category": b["category"],
        "section": b["section"],
        "chunk_index": b["chunk_index"],
        "language": b.get("language", "en")
    } for b in batch]

    embs = embedder.encode(docs, normalize_embeddings=True).tolist()
    collection.upsert(ids=ids, documents=docs, embeddings=embs, metadatas=metas)
    added += len(batch)

print(f"✅ Upserted {added} chunks into collection '{COLLECTION_NAME}'")
print("📦 Persist:", CHROMA_DIR.resolve())


Indexing to Chroma: 100%|██████████| 7/7 [06:11<00:00, 53.04s/it]

✅ Upserted 446 chunks into collection 'jenosize-ideas'
📦 Persist: D:\mini-jane-demo\notebooks\vectorstore


In [16]:
def search(query, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True).tolist()[0]
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "metadatas", "distances"]  # ⬅️ ตัด "ids" ออก
    )
    # ป้องกันเคสไม่มีผลลัพธ์
    if not res or not res.get("documents") or not res["documents"][0]:
        print("No results.")
        return
    
    # ids อาจยังถูกส่งกลับอยู่ แม้ไม่ได้ขอใน include; ถ้าไม่มี ก็ไม่แสดง
    ids = res.get("ids", [[]])[0] if res.get("ids") else [None] * len(res["documents"][0])

    for i, (doc, meta, dist, cid) in enumerate(
        zip(res["documents"][0], res["metadatas"][0], res["distances"][0], ids), 1
    ):
        print(f"\n[{i}] dist={dist:.4f} | {meta.get('title','(no title)')} ({meta.get('category','-')})"
              f" | section={meta.get('section','-')} #{meta.get('chunk_index','-')}")
        if cid: 
            print("id:", cid)
        print(meta.get('url','-'))
        print(doc[:200].replace("\n"," ") + " ...")

# ทดสอบ 2 ภาษา
print("---- TH ----")
search("แนวโน้ม AI สำหรับธุรกิจสมัยใหม่", k=5)

print("\n---- EN ----")
search("Benefits of Agentic AI in organizations", k=5)


---- TH ----

[1] dist=0.7369 | 8 Trending Startup Businesses for Modern Entrepreneurs (Experience the New World) | section=Body #5
id: https://www.jenosize.com/en/ideas/experience-the-new-world/top-startup-trends#chunk5
https://www.jenosize.com/en/ideas/experience-the-new-world/top-startup-trends
7. AI-Aided Engineering Tools The engineering tools for designing physical systems, from CAD/CAM to CFD software, have seen little advancement in decades. These tools often require complex simulations ...

[2] dist=0.7786 | Understanding 7 Megatrends for 2024 to Shape Your Tomorrow (Experience the New World) | section=Body #4
id: https://www.jenosize.com/en/ideas/experience-the-new-world/7-megatrends-in-2024#chunk4
https://www.jenosize.com/en/ideas/experience-the-new-world/7-megatrends-in-2024
For example, leading cosmetics and skincare companies are reformulating products to be organic and cruelty-free to meet the preferences of modern consumers, enabling organizations to sustainably grow. .

In [17]:
REPORT = PROJECT_ROOT / "data" / "processed" / "index_summary.txt"
REPORT.write_text(
    f"Collection: {COLLECTION_NAME}\nModel: {EMBED_MODEL}\nPersist: {CHROMA_DIR.resolve()}\nTotal inserted (this run): {added}\n",
    encoding="utf-8"
)
print("📝 Wrote:", REPORT)


📝 Wrote: D:\mini-jane-demo\data\processed\index_summary.txt
